## Аналіз A/B-тестів

Ви - аналітик даних в ІТ-компанії і до вас надійшла задача проаналізувати дані A/B тесту в популярній [грі Cookie Cats](https://www.facebook.com/cookiecatsgame). Це - гра-головоломка в стилі «з’єднай три», де гравець повинен з’єднати плитки одного кольору, щоб очистити дошку та виграти рівень. На дошці також зображені співаючі котики :)

Під час проходження гри гравці стикаються з воротами, які змушують їх чекати деякий час, перш ніж вони зможуть прогресувати або зробити покупку в додатку.

У цьому блоці завдань ми проаналізуємо результати A/B тесту, коли перші ворота в Cookie Cats було переміщено з рівня 30 на рівень 40. Зокрема, ми хочемо зрозуміти, як це вплинуло на утримання (retention) гравців. Тобто хочемо зрозуміти, чи переміщення воріт на 10 рівнів пізніше якимось чином вплинуло на те, що користувачі перестають грати в гру раніше чи пізніше з точки зору кількості їх днів з моменту встановлення гри.

Будемо працювати з даними з файлу `cookie_cats.csv`. Колонки в даних наступні:

- `userid` - унікальний номер, який ідентифікує кожного гравця.
- `version` - чи потрапив гравець в контрольну групу (gate_30 - ворота на 30 рівні) чи тестову групу (gate_40 - ворота на 40 рівні).
- `sum_gamerounds` - кількість ігрових раундів, зіграних гравцем протягом першого тижня після встановлення
- `retention_1` - чи через 1 день після встановлення гравець повернувся і почав грати?
- `retention_7` - чи через 7 днів після встановлення гравець повернувся і почав грати?

Коли гравець встановлював гру, його випадковим чином призначали до групи gate_30 або gate_40.

1. Для початку, уявімо, що ми тільки плануємо проведення зазначеного А/B-тесту і хочемо зрозуміти, дані про скількох користувачів нам треба зібрати, аби досягнути відчутного ефекту. Відчутним ефектом ми вважатимемо збільшення утримання на 1% після внесення зміни. Обчисліть, скільки користувачів сумарно нам треба аби досягнути такого ефекту, якщо продакт менеджер нам повідомив, що базове утримання є 19%.

Оскільки ми заздалегідь не знаємо, чи вплине перенесення воріт з 30-го на 40-й рівень на утримання гравців у бік збільшення чи зменшення, обираємо двосторонній тест:

$$H_0: p_1 = p_2$$
$$H_a: p_1 \ne p_2$$

де:

$p_1$ - частка утримання гравців у контрольній групі gate_30

$p_2$ - частка утримання гравців у тестовій групі gate_40

Також встановлюємо рівень значущості:
$$\alpha = 0.05$$

In [1]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.stats.api as sms
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from math import ceil

%matplotlib inline

In [2]:
effect_size = sms.proportion_effectsize(0.19, 0.20)

In [3]:
effect_size

np.float64(-0.025241594409087353)

In [5]:
required_n = sms.NormalIndPower().solve_power(
    effect_size,
    power=0.8,
    alpha=0.05,
    ratio=1
    )

required_n = ceil(required_n)

print(f"Users per group: {required_n}")
print(f"Total users needed: {required_n * 2}")

Users per group: 24638
Total users needed: 49276


Для виявлення збільшення retention з 19% до 20% при рівні значущості 0.05 та потужності 80% необхідно приблизно 24638 користувачів у кожній групі. Загальна кількість користувачів становить близько 49276.

2. Зчитайте дані АВ тесту у змінну `df` та виведіть середнє значення показника показник `retention_7` (утримання на 7 день) по версіям гри. Сформулюйте гіпотезу: яка версія дає краще утримання через 7 днів після встановлення гри?

In [8]:
df = pd.read_csv('../statistical_hypothesis/cookie_cats.csv')

df.head()

,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90189 entries, 0 to 90188
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   userid          90189 non-null  int64 
 1   version         90189 non-null  object
 2   sum_gamerounds  90189 non-null  int64 
 3   retention_1     90189 non-null  bool  
 4   retention_7     90189 non-null  bool  
dtypes: bool(2), int64(2), object(1)
memory usage: 2.2+ MB


In [12]:
df.groupby('version')['retention_7'].mean()

version
gate_30    0.190201
gate_40    0.182000
Name: retention_7, dtype: float64

Середнє значення retention_7 показує частку користувачів, які повернулися в гру через 7 днів у кожній версії.

Оскільки ми не знаємо наперед, яка версія дає краще утримання, використовуємо двосторонній тест:

$$H_0: p_1 = p_2$$
$$H_a: p_1 \ne p_2$$

де:

$p_1$ - частка користувачів, які повернулися через 7 днів у версії gate_30

$p_2$ - частка користувачів, які повернулися через 7 днів у версії gate_40

Гіпотеза залишається такою ж, як і на етапі планування експерименту.

3. Перевірте з допомогою пасуючого варіанту z-тесту, чи дає якась з версій гри кращий показник `retention_7` на рівні значущості 0.05. Обчисліть також довірчі інтервали для варіантів до переміщення воріт і після. Виведіть результат у форматі:

    ```
    z statistic: ...
    p-value: ...
    Довірчий інтервал 95% для групи control: [..., ...]
    Довірчий інтервал 95% для групи treatment: [..., ...]
    ```

    де замість `...` - обчислені значення.
    
    В якості висновку дайте відповідь на два питання:  

      1. Чи є статистична значущою різниця між поведінкою користувачів у різних версіях гри?   
      2. Чи перетинаються довірчі інтервали утримання користувачів з різних версій гри? Про що це каже?  


In [14]:
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

In [18]:
control_results = df[df['version'] == 'gate_30']['retention_7']
treatment_results = df[df['version'] == 'gate_40']['retention_7']

In [19]:
n_con = control_results.count()
n_treat = treatment_results.count()
successes = [control_results.sum(), treatment_results.sum()]
nobs = [n_con, n_treat]

In [20]:
successes

[np.int64(8502), np.int64(8279)]

In [21]:
z_stat, pval = proportions_ztest(successes, nobs=nobs)
(lower_con, lower_treat), (upper_con, upper_treat) = proportion_confint(successes, nobs=nobs, alpha=0.05)

print(f'z statistic: {z_stat:.2f}')
print(f'p-value: {pval:.3f}')
print(f'Довірчий інтервал 95% для групи control: [{lower_con:.3f}, {upper_con:.3f}]')
print(f'Довірчий інтервал 95% для групи treatment: [{lower_treat:.3f}, {upper_treat:.3f}]')

z statistic: 3.16
p-value: 0.002
Довірчий інтервал 95% для групи control: [0.187, 0.194]
Довірчий інтервал 95% для групи treatment: [0.178, 0.186]


A. Оскільки наше $p$-значення=0.002 є значно меншим за рівень значущості $\alpha$ = 0.05, ми відхиляємо нульову гіпотезу $H_0$. Це означає, що існує статистично значуща різниця між поведінкою користувачів у різних версіях гри.

B. Крім того, якщо ми подивимося довірчі інтервали:

* для групи `control` [0,187, 0,194], тобто 18,7-19,4%
* для групи `treatment` [0,178, 0,186], тобто 17,8-18,6%

ми помітимо, що:
1. Ці інтервали не перетинаються між собою.
2. Інтервал для control (gate_30) повністю лежить вище, ніж для treatment (gate_40).

Це означає, що справжній рівень утримання користувачів у версії gate_30 є вищим, ніж у версії gate_40.

Таким чином, результати довірчих інтервалів підтверджують висновок статистичного тесту: перенесення воріт на 40-й рівень негативно вплинуло на retention і версія gate_30 показує кращий результат.

4. Виконайте тест Хі-квадрат на рівні значущості 5% аби визначити, чи є залежність між версією гри та утриманням гравця на 7ий день після реєстрації.

    - Напишіть, як для цього тесту будуть сформульовані гіпотези.
    - Проведіть обчислення, виведіть p-значення і напишіть висновок за результатами тесту.


**Гіпотези:**

* $H_0$: перенесення воріт не впливає на утримання гравців.
  
* $H_a$: перенесення воріт впливає на утримання гравців.

In [26]:
from scipy.stats import chi2_contingency

In [27]:
contingency_table = pd.crosstab(df['version'], df['retention_7'])

In [29]:
chi2, p, dof, expected = stats.chi2_contingency(contingency_table)

print(f"χ² = {chi2:.3f}")
print(f"p-value = {p:.5f}")
print(f"Ступені свободи = {dof}")
print("Очікувані частоти:\n", expected)

χ² = 9.959
p-value = 0.00160
Ступені свободи = 1
Очікувані частоти:
 [[36382.90257127  8317.09742873]
 [37025.09742873  8463.90257127]]


Оскільки наше $p$-значення=0.002 є значно меншим за рівень значущості $\alpha$ = 0.05, ми відхиляємо нульову гіпотезу $H_0$. Це означає, що існує статистично значуща залежність між версією гри та утриманням користувачів на 7-й день.

Це свідчить, що зміна розташування воріт (з 30-го на 40-й рівень) впливає на поведінку користувачів.

Таким чином, результати χ²-тесту підтверджують, що версія гри є важливим фактором, який впливає на retention користувачів.

Це узгоджується з результатами попереднього z-тесту, де також було виявлено статистично значущу різницю між групами.